In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ml2022spring-hw2/sample_submission.csv
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/train_split.txt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/train_labels.txt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/test_split.txt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/374-180299-0035.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/4195-186237-0009.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/730-360-0036.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/4441-76262-0000.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/6019-3185-0107.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/4051-11217-0019.pt
/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/test/6181-216552-0001.pt
/kaggle/input/competition

In [2]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'当前使用的设备是: {device}')


当前使用的设备是: cuda


### 花名册-标签-帧关系
#### 使用花名册得到要处理的帧文件与对应的标签文件
#### 帧文件的内容需要11拼接拼接后为x标签文件为y_hat(训练集的结果)

In [4]:
path_split='/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/train_split.txt'
path_labels='/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/train_labels.txt'
path_feat='/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/train'
#path4='/kaggle/input/competitions/ml2022spring-hw2/libriphone/libriphone/feat/train/4051-11217-0006.pt'
with open(path_split,'r') as f:
    usage_list=[line.strip('\n')for line in f.readlines()]




    

In [5]:
def shift(x,n):
    if(n<0):
        left=x[0].repeat(-n,1)
        right=x[:n]
        return torch.cat((left,right),dim=0)
    elif n>0:
        left=x[n:]
        right=x[-1].repeat(n,1)
        return torch.cat((left,right),dim=0)
    else:
        return x

In [6]:
#拼接函数
def concat_feat(x,concat_n=11):
    #特殊情况第一帧与第15帧：重复
    seq_len,feature_dim=x.size(0),x.size(1)#特征数量是列数
    #复制11份
    x=x.repeat(1,concat_n)
    #修改维度为三维(方便移动)
    x=x.view(seq_len,concat_n,feature_dim)
    mid=5
    for r_idx in range(1,6):
        x[:,5+r_idx,:]=shift(x[:,mid+r_idx,:],r_idx)
        x[:,5-r_idx,:]=shift(x[:,mid-r_idx,:],-r_idx)
    return x.view(seq_len,concat_n*feature_dim)
    

In [7]:
label_dict = {}
with open(path_labels, 'r') as f:
    for line in f.readlines():
        parts = line.strip('\n').split(' ')
        label_dict[parts[0]] = [int(p) for p in parts[1:]]
X,y=[],[]
#使用循环处理每一个花名册-标签-帧的关系
for fname in usage_list:
    feat=torch.load(f'{path_feat}/{fname}.pt')
    label=label_dict[fname]
    concat_feature=concat_feat(feat,concat_n=11)

    X.append(concat_feature)#每次循环中标签有多个所以使用append
    y.extend(label)#label是一个对象使用extend




In [34]:
X


array([tensor([[-2.0803, -0.7538, -0.4708,  ...,  0.2271, -0.1655,  0.2518],
               [-2.0803, -0.7538, -0.4708,  ...,  0.4836, -0.6296, -0.3662],
               [-2.0803, -0.7538, -0.4708,  ...,  0.2184, -0.6470, -0.3058],
               ...,
               [-1.8150, -0.6722, -0.4825,  ..., -0.6997, -1.5800, -1.4590],
               [-1.7635, -0.6816, -0.5401,  ..., -0.6997, -1.5800, -1.4590],
               [-1.7944, -0.9144, -0.5468,  ..., -0.6997, -1.5800, -1.4590]]),
       tensor([[-1.1812, -0.5272, -0.5843,  ..., -0.3071, -0.9441, -1.2543],
               [-1.1812, -0.5272, -0.5843,  ...,  0.2129, -0.0406, -0.3076],
               [-1.1812, -0.5272, -0.5843,  ...,  0.9274,  1.0964,  0.6840],
               ...,
               [-0.6908, -1.2008, -0.8424,  ...,  0.2892,  0.1331, -0.2891],
               [-0.9575, -0.4410, -0.5832,  ...,  0.2892,  0.1331, -0.2891],
               [-1.1587, -0.4242, -0.4060,  ...,  0.2892,  0.1331, -0.2891]]),
       tensor([[-1.9556, -0.2677

## 若直接将得到的x,y转化为张量则会爆内存
### 解决方法使用自定义的Dataset直接处理x_list、y_list（已经证明失效）
### 采用cat合并X的list中tensor  合并过程中会不断复制已经合并的中间结果x随着x增大效率减慢
### 采用预分配先定义最终结果大小的tensor再不断将x每一个tensor复制进去

#### 2.cat合并

In [35]:
'''

y=np.array(y)
X=pd.Series(X)
X=X.to_numpy()
for i in range(X.shape[0]):
    if i==0:
        x=X[0]
    if i!=0:
        x=torch.cat([x,X[i]],dim=0)  #败笔：使用cat每次都要加载x，随着x越来越大，cpu会爆红
        
x_train=x
y_train=torch.tensor(y)
print(x_train.shape,y_train.shape)
train_dataset=TensorDataset(x_train,y_train)
train_dataloader=DataLoader(train_dataset,batch_size=10000,shuffle=True)
x_train.shape,y_train.shape

SyntaxError: incomplete input (1027907648.py, line 1)

#### 1. 自定义的dataset:
自定义的dataset本质上在做将每个x向量与对应的y标签值返回，不过要注意继承自torch的dataset，函数名要严格一致

In [25]:
'''
import bisect
import torch
from torch.utils.data import Dataset, DataLoader

#自定义的dataset本质上在做将每个x向量与对应的y标签值返回

class MemorySafeFastDataset(Dataset):
    def __init__(self, x_list, y_list): #初始化函数接收实参计算累计长度
        self.x = x_list
        self.y = y_list
        
        # 1. 预计算累积长度（纯 Python 列表，占用极小）
        self.cumulative_lengths = [0]
        for feat in x_list:            #feat是tensor类型
            self.cumulative_lengths.append(self.cumulative_lengths[-1] + feat.shape[0])
            #【-1】cumulative_len的最后一个值再加上 新的tensor的长度得到一个新的长度位置代表每一个结束位置
            # [0,len1,len1+len2,len2+len3...]这样的累积长度列表
    
    def __len__(self):     #计算最终长度
        return self.cumulative_lengths[-1] #得到累计长度列表中最后一个元素  [-1]直接得到最后一个元素
    
    def __getitem__(self, idx):
        # 🔥 核心优化 1：使用 Python 底层 C 实现的二分查找，O(log N)，极快且无 .item() 开销
        file_idx = bisect.bisect_right(self.cumulative_lengths, idx) - 1  #tensor索引
        
        # 🔥 核心优化 2：纯 Python 整数运算计算局部索引
        local_idx = idx - self.cumulative_lengths[file_idx]# tensor内部行索引
        
        # 返回数据（DataLoader 会自动将其打包为 Tensor）
        return self.x[file_idx][local_idx], self.y[file_idx]  #返回一行x一个y刚好对应训练x向量 与y标签值

# 创建 Dataset
train_dataset = MemorySafeFastDataset(X, y)

# 🔥 核心优化 3：DataLoader 配置
train_loader = DataLoader(
    train_dataset,
    batch_size=10000,       # 1. 调大 Batch Size，摊薄 CPU 固定开销
    shuffle=True,
    num_workers=0,        # 2. 【关键】设为 0！纯内存数据+大List，单线程直读最快，避免多进程序列化
    pin_memory=True,      # 3. 保持开启，加速 CPU->GPU 传输
)

#### 3.预分配法

In [56]:
from torch.utils.data import Dataset, DataLoader 
import gc

x_list = X  # 可能是 list of Tensor 或 numpy 数组 (dtype=object)
total_rows = sum(t.shape[0] for t in x_list)
x_train = torch.empty((total_rows, 429), dtype=torch.float32)

start_idx = 0
batch_size = 10000   # 每次处理的子张量个数

for i in range(0, len(x_list), batch_size):
    end = min(i + batch_size, len(x_list))
    batch_tensors = x_list[i:end]          # 可能是 list 或 numpy 数组
    
    # 兼容处理：如果是 numpy 数组，转为 list
    if isinstance(batch_tensors, np.ndarray):
        batch_tensors = batch_tensors.tolist()
    
    stacked = torch.cat(batch_tensors, dim=0)
    rows = stacked.shape[0]
    x_train[start_idx:start_idx+rows] = stacked
    start_idx += rows

    del stacked, batch_tensors

# 释放原始数据
del x_list
gc.collect()


y=np.array(y)
y_train=torch.tensor(y)
train_dataset=TensorDataset(x_train,y_train)
train_dataloader=DataLoader(train_dataset,batch_size=10000,shuffle=True)

NameError: name 'TensorDataset' is not defined

In [ ]:
#定义神经网络结构
class Multisort(torch.nn.Module):
    def __init__(self):
        super(Multisort,self).__init__()

        self.layer1=torch.nn.Linear(429,128)
        self.ReLu=torch.nn.ReLU(inplace=False)
        self.layer2=torch.nn.Linear(128,64)
        self.ReLu=torch.nn.ReLU(inplace=False)
        self.layer3=torch.nn.Linear(64,41)


    def forward(self,x):
        x=self.layer1(x)
        x=self.ReLu(x)
        x=self.layer2(x)
        x=self.ReLu(x)
        x=self.layer3(x)
        return x

model=Multisort()
model=model.to(device)

critrision=torch.nn.CrossEntropyLoss(reduction='mean')
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)

In [57]:
print(f"Model device: {next(model.parameters()).device}") 
print(f"Data device: {device}")

Model device: cuda:0
Data device: cuda


In [59]:
import time

print("开始测试数据加载速度...")
start_time = time.time()

# 只跑一个epoch的数据加载
for i, (batch_x, batch_y) in enumerate(train_loader):
    if i % 100 == 0:
        print(f"已加载 {i} 个batch")
    # 只加载，不进行任何模型计算
    pass 

end_time = time.time()
print(f"加载完整个数据集耗时: {end_time - start_time:.2f} 秒")

开始测试数据加载速度...
已加载 0 个batch
已加载 100 个batch
已加载 200 个batch
加载完整个数据集耗时: 21.22 秒


In [ ]:
for epoch in range(1001):
    model.train()
    epoch_loss = 0.0  # 用于累积每个 epoch 的 loss
    
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        # 1. 数据传输
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        
        # 2. 梯度清零 (放在最前面！)
        optimizer.zero_grad()
        
        # 3. 前向传播
        y_pred = model(batch_x)
        loss = critrision(y_pred, batch_y)
        
        # 4. 反向传播
        loss.backward()
        
        # 5. 更新参数
        optimizer.step()
        
        # 6. 累积 loss (只取数值，不打印)
        epoch_loss += loss.item()
        
        # 7. 每隔 N 个 batch 打印一次 (可选，但不要每个都打印)
        # if batch_idx % 100 == 0:
        #     print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')
    
    # 8. 每个 epoch 结束后，打印平均 loss
    print(f'Epoch {epoch}, Average Loss: {epoch_loss / len(train_loader):.4f}')

Epoch 0, Average Loss: 3.1280
Epoch 1, Average Loss: 3.1277
Epoch 2, Average Loss: 3.1276
Epoch 3, Average Loss: 3.1273
Epoch 4, Average Loss: 3.1271
Epoch 5, Average Loss: 3.1269
Epoch 6, Average Loss: 3.1268
Epoch 7, Average Loss: 3.1266
Epoch 8, Average Loss: 3.1265
Epoch 9, Average Loss: 3.1263
Epoch 10, Average Loss: 3.1262
Epoch 11, Average Loss: 3.1260
Epoch 12, Average Loss: 3.1259
Epoch 13, Average Loss: 3.1258
Epoch 14, Average Loss: 3.1255
Epoch 15, Average Loss: 3.1255
Epoch 16, Average Loss: 3.1254


In [58]:
for i in range(1001):
    model.train()
    for batch_x,batch_y in train_loader:
        batch_x=batch_x.to(device,non_blocking=True)
        batch_y=batch_y.to(device,non_blocking=True)
       # print(f"batch_x device: {batch_x.device}, batch_y device: {batch_y.device}")
        y_pred=model(batch_x)
        loss=critrision(y_pred,batch_y)
        if(i%200==0):
            print(f'loop:{i},Loss is{loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
   
    

loop:0,Loss is3.148829221725464
loop:0,Loss is3.160857915878296
loop:0,Loss is3.1503751277923584
loop:0,Loss is3.1616249084472656
loop:0,Loss is3.152820587158203
loop:0,Loss is3.1590211391448975
loop:0,Loss is3.1618237495422363
loop:0,Loss is3.16500186920166
loop:0,Loss is3.1578454971313477
loop:0,Loss is3.1643729209899902
loop:0,Loss is3.175281524658203
loop:0,Loss is3.159776449203491
loop:0,Loss is3.153646945953369
loop:0,Loss is3.1541008949279785
loop:0,Loss is3.160922050476074
loop:0,Loss is3.1463770866394043
loop:0,Loss is3.15545916557312
loop:0,Loss is3.16755747795105
loop:0,Loss is3.179654359817505
loop:0,Loss is3.1602602005004883
loop:0,Loss is3.1628010272979736
loop:0,Loss is3.150282859802246
loop:0,Loss is3.147141695022583
loop:0,Loss is3.1612801551818848
loop:0,Loss is3.148249387741089
loop:0,Loss is3.1673169136047363
loop:0,Loss is3.15415358543396
loop:0,Loss is3.1514217853546143
loop:0,Loss is3.1468961238861084
loop:0,Loss is3.1530821323394775
loop:0,Loss is3.1393144130706

KeyboardInterrupt: 

In [ ]:
!nvidia-smi


In [ ]:
torch.save(model, '/kaggle/working/full_model.pth')